# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PrasannaSaiS/machinelearning01-flyrank/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card."


## 1. Two paper findings + my methodology questions

1. Finding: the strongest predictive patterns concern visibility and content depth, while engagement depth is weak or flat. This is a useful and well-scoped claim only if the label is measured in a future window and not reconstructed from the same period.

   My methodology question: where is the label defined, and does the paper allow the feature window to precede the label window without overlap?

2. Finding: the queue-style model is best interpreted as decision-support, not as a claim that a page will definitely be cited or win. This is stronger and safer language than a direct outcome claim.

   My methodology question: are ranking metrics computed on a held-out client/time-aware split and reported against a naive baseline so the lift is interpretable rather than only impressive?

The spirit of the audit is constructive: I am checking whether the label, population, and split actually make the claim credible in a real deployment setting.


In [3]:
import os
from pathlib import Path
import pandas as pd

# 1. Clone repository if data folder is not found in Colab workspace
repo_url = "https://github.com/PrasannaSaiS/machinelearning01-flyrank.git"
repo_dir = Path("machinelearning01-flyrank")

if not Path("data").exists() and not (repo_dir / "data").exists():
    !git clone {repo_url}

if repo_dir.exists():
    os.chdir(repo_dir)

repo = Path.cwd().resolve()

# 2. Build the processed feature dataset if it doesn't exist yet
data_path = repo / 'data' / 'processed' / 'refresh_feature_vector.csv'
if not data_path.exists() and (repo / 'scripts' / '01_prepare_features.py').exists():
    !python scripts/01_prepare_features.py

# 3. Read processed dataset
df = pd.read_csv(data_path)

## 2. My model under an honest split (before/after)

A random row-level split often overstates performance because pages from the same client can appear in both train and test. The honest deployment check is a grouped split by `client_id` or a time-aware split.


In [5]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupKFold, train_test_split

repo = Path.cwd().resolve()
if not (repo / 'data').exists():
    for candidate in repo.parents:
        if (candidate / 'data').exists() and (candidate / 'scripts').exists():
            repo = candidate
            break

df = pd.read_csv(repo / 'data' / 'processed' / 'refresh_feature_vector.csv')
numeric_features = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d',
    'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
]
categorical_features = [
    'competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier',
    'word_count_tier', 'impression_tier', 'position_tier',
]

numeric_frame = df[numeric_features].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
categorical_frame = df[categorical_features].fillna('unknown').astype(str)
X = pd.concat([numeric_frame.reset_index(drop=True), pd.get_dummies(categorical_frame, prefix=categorical_features, dummy_na=False, dtype=float).reset_index(drop=True)], axis=1)
y = df['is_declining_label'].astype(int)

def precision_at_k(y_true, scores, k):
    frame = pd.DataFrame({'y': list(y_true), 'score': list(scores)})
    frame = frame.sort_values('score', ascending=False)
    top = frame.head(min(k, len(frame)))
    return float(top['y'].mean()) if len(top) else 0.0

idx_train, idx_test = train_test_split(np.arange(len(df)), test_size=0.2, random_state=42, stratify=y)
random_model = RandomForestClassifier(class_weight='balanced_subsample', max_depth=10, min_samples_leaf=25, n_estimators=200, random_state=42)
random_model.fit(X.iloc[idx_train], y.iloc[idx_train])
random_probs = random_model.predict_proba(X.iloc[idx_test])[:, 1]
random_metrics = {
    'precision_at_50': precision_at_k(y.iloc[idx_test], random_probs, 50),
    'roc_auc': roc_auc_score(y.iloc[idx_test], random_probs),
    'average_precision': average_precision_score(y.iloc[idx_test], random_probs),
    'base_rate': float(y.iloc[idx_test].mean()),
}

fold_scores = []
for tr_idx, te_idx in GroupKFold(n_splits=5).split(X, y, groups=df['client_id']):
    honest_model = RandomForestClassifier(class_weight='balanced_subsample', max_depth=10, min_samples_leaf=25, n_estimators=200, random_state=42)
    honest_model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    honest_probs = honest_model.predict_proba(X.iloc[te_idx])[:, 1]
    fold_scores.append({
        'precision_at_50': precision_at_k(y.iloc[te_idx], honest_probs, 50),
        'roc_auc': roc_auc_score(y.iloc[te_idx], honest_probs),
        'average_precision': average_precision_score(y.iloc[te_idx], honest_probs),
        'base_rate': float(y.iloc[te_idx].mean()),
    })

honest_summary = {metric: float(np.mean([s[metric] for s in fold_scores])) for metric in fold_scores[0]}
print('Random holdout metrics:', {k: round(v, 3) for k, v in random_metrics.items()})
print('Grouped-by-client honest metrics (mean across folds):', {k: round(v, 3) for k, v in honest_summary.items()})
print('Fold-level honest metrics:', [{k: round(float(v), 3) for k, v in s.items()} for s in fold_scores])
print('Interpretation: the honest split is the realistic deployment check; random row-level leakage tends to overstate performance.')


Random holdout metrics: {'precision_at_50': 0.9, 'roc_auc': np.float64(0.758), 'average_precision': np.float64(0.768), 'base_rate': 0.542}
Grouped-by-client honest metrics (mean across folds): {'precision_at_50': 0.716, 'roc_auc': 0.666, 'average_precision': 0.67, 'base_rate': 0.544}
Fold-level honest metrics: [{'precision_at_50': 0.86, 'roc_auc': 0.654, 'average_precision': 0.636, 'base_rate': 0.49}, {'precision_at_50': 0.92, 'roc_auc': 0.611, 'average_precision': 0.733, 'base_rate': 0.645}, {'precision_at_50': 0.72, 'roc_auc': 0.738, 'average_precision': 0.589, 'base_rate': 0.379}, {'precision_at_50': 0.66, 'roc_auc': 0.66, 'average_precision': 0.706, 'base_rate': 0.622}, {'precision_at_50': 0.42, 'roc_auc': 0.666, 'average_precision': 0.686, 'base_rate': 0.585}]
Interpretation: the honest split is the realistic deployment check; random row-level leakage tends to overstate performance.


## 3. Leakage audit

The right test is not "can I get a high number?" It is "does a feature that directly defines the target cause the score to jump unexpectedly?" If the answer is yes, the feature is leaking.


In [6]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split

repo = Path.cwd().resolve()
if not (repo / 'data').exists():
    for candidate in repo.parents:
        if (candidate / 'data').exists() and (candidate / 'scripts').exists():
            repo = candidate
            break

df = pd.read_csv(repo / 'data' / 'processed' / 'refresh_feature_vector.csv')
numeric_features = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d',
    'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
]
categorical_features = [
    'competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier',
    'word_count_tier', 'impression_tier', 'position_tier',
]

numeric_frame = df[numeric_features].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
categorical_frame = df[categorical_features].fillna('unknown').astype(str)
base_matrix = pd.concat([numeric_frame.reset_index(drop=True), pd.get_dummies(categorical_frame, prefix=categorical_features, dummy_na=False, dtype=float).reset_index(drop=True)], axis=1)

def precision_at_k(y_true, scores, k):
    frame = pd.DataFrame({'y': list(y_true), 'score': list(scores)})
    frame = frame.sort_values('score', ascending=False)
    top = frame.head(min(k, len(frame)))
    return float(top['y'].mean()) if len(top) else 0.0

def evaluate(feature_frame):
    y = df['is_declining_label'].astype(int)
    idx_train, idx_test = train_test_split(np.arange(len(df)), test_size=0.2, random_state=42, stratify=y)
    model = RandomForestClassifier(class_weight='balanced_subsample', max_depth=10, min_samples_leaf=25, n_estimators=200, random_state=42)
    model.fit(feature_frame.iloc[idx_train], y.iloc[idx_train])
    probs = model.predict_proba(feature_frame.iloc[idx_test])[:, 1]
    return {
        'precision_at_50': precision_at_k(y.iloc[idx_test], probs, 50),
        'roc_auc': roc_auc_score(y.iloc[idx_test], probs),
        'average_precision': average_precision_score(y.iloc[idx_test], probs),
        'base_rate': float(y.iloc[idx_test].mean()),
    }

safe_metrics = evaluate(base_matrix)
leaky_matrix = base_matrix.copy()
leaky_matrix['trend_pct'] = df['trend_pct'].fillna(0).astype(float)
leaky_matrix['trend_direction_down'] = (df['trend_direction'].fillna('unknown').str.lower() == 'down').astype(int)
leaky_metrics = evaluate(leaky_matrix)
print('Safe feature set:', {k: round(v, 3) if isinstance(v, float) else v for k, v in safe_metrics.items()})
print('Leaky feature set:', {k: round(v, 3) if isinstance(v, float) else v for k, v in leaky_metrics.items()})
print('Interpretation: if adding a label-defining signal causes a large jump, that signal is leaking and should be excluded from any final claim.')


Safe feature set: {'precision_at_50': 0.9, 'roc_auc': np.float64(0.758), 'average_precision': np.float64(0.768), 'base_rate': 0.542}
Leaky feature set: {'precision_at_50': 1.0, 'roc_auc': np.float64(1.0), 'average_precision': np.float64(1.0), 'base_rate': 0.542}
Interpretation: if adding a label-defining signal causes a large jump, that signal is leaking and should be excluded from any final claim.


## 4. Claim rewrite

The honest rewrite is not "this page definitely declines." It is "in this sample, the model measured a directional lift over a naive baseline and is best used as decision-support for prioritization."


In [7]:
bold_claim = 'This model can identify the pages most likely to decline and should be used to prioritize review work.'
safe_claim = (
    'In the prepared sample, the model measured a directional lift over a naive baseline for the observed decline label. '
    'It is best interpreted as decision-support for prioritizing review queues, not as a guarantee that a page will decline or be cited.'
)
print('Bold claim:', bold_claim)
print('Safe rewrite:', safe_claim)


Bold claim: This model can identify the pages most likely to decline and should be used to prioritize review work.
Safe rewrite: In the prepared sample, the model measured a directional lift over a naive baseline for the observed decline label. It is best interpreted as decision-support for prioritizing review queues, not as a guarantee that a page will decline or be cited.


# Self-check
Before you submit, confirm each line honestly:

* Every section above is filled — markdown thinking AND the code that backs it
* The notebook runs top to bottom with no errors (Runtime → Run all)
* No client names, URLs, or private queries anywhere
* My claims use careful words: observed, measured, directional, decision-support
* Committed to my repo under 'work/notebooks/' — then submit your repo URL on the card. Done.